In [ ]:
import flopy
import pyemu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
from pathlib import Path

# Intro

This notebook runs a prior monte carlo using the pest setup generated in the `setup_pstfrom` notebook. We will run the prior parameter ensemble once and stop. We will take a look at some of the results.

### Warning
This notebook can take some time to run. That is the cost of a >5min model run time...imagine if your model takes longer than that (#suffering). For context, this takes about 60min on a MacBook Pro. Expect longer on Windows. We recommend setting this to run over night or when you go out for lunch, before proceeding to the next notebooks. 

In [ ]:
# specify the temporary working folder
t_d = Path('priormc')
# get the previously generated PEST dataset
org_t_d = Path('pst_template')
if not Path.exists(org_t_d):
    raise Exception()
if Path.exists(t_d):
    shutil.rmtree(t_d)
shutil.copytree(org_t_d,t_d)

# Prior Monte Carlo

Load the pest control file

In [ ]:
pst = pyemu.Pst(str(t_d/"pest.pst"))

Lets run the full 1000 realizations. (Why will become evident when we reach the `dsi-ae` notebook...)

In [ ]:
pst.pestpp_options["ies_num_reals"] = 20
pst.pestpp_options["save_binary"] = True

Set noptmax=-1 to just run the ensemble once. And lets store results in binary format for speed and disk savings...

In [ ]:
pst.control_data.noptmax = -1
pst.write(Path(t_d,"pest.pst"),version=2)

### Warning: set num_workers according to your reserouces

The model took ~6 min to run... so that would be about 10h for the full 100 runs in serial. Gonna need to run them in parallel to be reasonable. 

With 10 agents we should be looking at about 1.6h instead..a bit more manageable. 

In [ ]:
num_workers = 10
m_d = "master_priormc"

In [ ]:

pyemu.os_utils.start_workers(t_d, # the folder which contains the "template" PEST dataset
                            'pestpp-ies', #the PEST software version we want to run
                            'pest.pst', # the control file to use with PEST
                            num_workers=num_workers, #how many agents to deploy
                            worker_root='.', #where to deploy the agent directories; relative to where python is running
                            master_dir=m_d, #the manager directory
                            )

## Load results

In [ ]:
pst = pyemu.Pst(str(Path(m_d,"pest.pst")))

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=m_d, load_only=[],verbosity_level=0)
gwf = sim.get_model('gwf')

In [ ]:
obs = pst.observation_data
obs.variable.unique()

In [ ]:
obs.obsid.unique()

In [ ]:
obs.time

In [ ]:
obs.obsnme

In [ ]:
oe = pst.ies.obsen.copy()
oe.head()

## Make some figures

Lets look at some of the simualted max temperature fields.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

obs_to_plot = ['welopt-ly3', 'wp1-f3', 'wp4-f5']
variables = sorted(obs["variable"].unique())

meas_data = pd.read_csv(t_d / "_obs.conc.simvsmeas.csv")
meas_data['variable'] = meas_data.variable.str.lower()
meas_data = meas_data[meas_data["meas"] <= 1e30]

def plot_var(var_to_plot):
    if var_to_plot=='tmp':
        c=1e3
    else:
        c=1
    fig, axs = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(10, 2))

    for ax, oid in zip(axs.flatten(), obs_to_plot ):
        tmp = obs[(obs.obsid == oid) & (obs.variable == var_to_plot)].copy()
        tmp['time'] = tmp.time.astype(float)
        tmp.sort_values(by='time', inplace=True)
        obnmes = tmp.obsnme.values
        [ax.plot(tmp.time.values, oe.loc[i, obnmes]*c, color = '0.5', alpha=0.5) for i in oe.index];
        ax.plot(tmp.time.values, oe.loc['base', obnmes]*c, label="base", c='b')

        # plot measured data
        tmp_meas = meas_data[(meas_data.obsid == oid) & (meas_data.variable == var_to_plot)].copy()
        ax.scatter(tmp_meas.time, tmp_meas.meas*c, s=20,
                   label="measured", edgecolor='r', facecolor='none',
                   zorder=10)
        ax.set_title(oid.upper())
        ax.set_ylabel(var_to_plot)

    # axs[0].legend(loc='lower right')
    fig.tight_layout()
    plt.show()

dropdown = widgets.Dropdown(
    options=variables,
    value="so4",
    description="Variable:",
)

widgets.interact(plot_var, var_to_plot=dropdown);

In [ ]:
import ipywidgets as widgets
from IPython.display import display

obs_to_plot = ['welopt-ly3', 'wp1-f3', 'wp4-f5']
variables = sorted(obs["variable"].unique())

meas_data = pd.read_csv(t_d / "_obs.conc.simvsmeas.csv")
meas_data['variable'] = meas_data.variable.str.lower()
meas_data = meas_data[meas_data["meas"] <= 1e30]

def plot_var(var_to_plot, pct=0.1):
    """
    Plot variable for each obsid with y-axis limits based on a percentage of measured values.
    
    Parameters:
        var_to_plot : str
            The variable to plot.
        pct : float
            Fractional margin to expand y-axis limits (default 0.1 = ±10%).
    """
    if var_to_plot == 'tmp':
        c = 1e3
    else:
        c = 1

    fig, axs = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(10, 2))  # sharey=False for individual limits

    for ax, oid in zip(axs.flatten(), obs_to_plot):
        tmp = obs[(obs.obsid == oid) & (obs.variable == var_to_plot)].copy()
        tmp['time'] = tmp.time.astype(float)
        tmp.sort_values(by='time', inplace=True)
        obnmes = tmp.obsnme.values
        [ax.plot(tmp.time.values, oe.loc[i, obnmes]*c, color='0.5', alpha=0.5) for i in oe.index]
        ax.plot(tmp.time.values, oe.loc['base', obnmes]*c, label="base", c='b')

        # plot measured data
        tmp_meas = meas_data[(meas_data.obsid == oid) & (meas_data.variable == var_to_plot)].copy()
        ax.scatter(tmp_meas.time, tmp_meas.meas*c, s=20,
                   label="measured", edgecolor='r', facecolor='none',
                   zorder=10)
        
        # set y-axis limits based on measured values ± pct
        if not tmp_meas.empty:
            y_min = tmp_meas.meas.min() * (1 - pct)
            y_max = tmp_meas.meas.max() * (1 + pct)
            ax.set_ylim(y_min * c, y_max * c)

        ax.set_title(oid.upper())
        ax.set_ylabel(var_to_plot)

    fig.tight_layout()
    plt.show()


dropdown = widgets.Dropdown(
    options=variables,
    value="so4",
    description="Variable:",
)

widgets.interact(plot_var, var_to_plot=dropdown, pct=0.5);